# smart_open: 5 Basic Usage Examples

In [ ]:
# Example 1: Open a local file
import os
import tempfile
from smart_open import open

# Keep the demo files out of the repo test dir: a notebook-scoped temp dir, kept alive
# across cells, so a partial run (e.g. the network example below failing) leaves no litter.
_work = tempfile.TemporaryDirectory(prefix="smart_open_")
txt_path = os.path.join(_work.name, "test.txt")
with open(txt_path, "w") as f:
    f.write('hello world')
with open(txt_path, "r") as f:
    print(f.read())


In [ ]:
# Example 2: Open a file over HTTP (read-only) — served by a local stdlib HTTP server,
# so the cell exercises smart_open's http transport without external network dependence.
import functools
import http.server
import threading
from smart_open import open

http_txt = os.path.join(_work.name, "served.txt")
with open(http_txt, "w") as f:
    f.write("hello over http\nsecond line\n")

handler = functools.partial(http.server.SimpleHTTPRequestHandler, directory=_work.name)
server = http.server.ThreadingHTTPServer(("127.0.0.1", 0), handler)
threading.Thread(target=server.serve_forever, daemon=True).start()

url = f"http://127.0.0.1:{server.server_address[1]}/served.txt"
with open(url) as f:
    first_line = f.readline()
server.shutdown()

print(f"read via {url}: {first_line!r}")
assert first_line == "hello over http\n"

In [ ]:
# Example 3: Open a gzip file
from smart_open import open
import gzip
gz_path = os.path.join(_work.name, "test.gz")
with gzip.open(gz_path, "wb") as f:
    f.write(b'compressed data')
with open(gz_path, "rb") as f:
    print(f.read())


In [ ]:
# Example 4: Clean up test files
_work.cleanup()
print('Removed test.txt and test.gz')


In [ ]:
# Example 5: Show smart_open version
import smart_open
print(smart_open.__version__)
